In [176]:
from dataclasses import dataclass

import pandas as pd
import numpy as np

In [177]:
data = pd.read_csv('marketing_campaign_dataset.csv')
print(f'dataset loaded with {data.shape[0]} rows and {data.shape[1]} columns')

dataset loaded with 2020 rows and 12 columns


In [178]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2020 entries, 0 to 2019
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0    Campaign_ID   2020 non-null   str    
 1   Campaign_Name  2020 non-null   str    
 2   Start_Date     2020 non-null   str    
 3   End_Date       2020 non-null   str    
 4   Channel        1919 non-null   str    
 5   Impressions    2020 non-null   int64  
 6   Clicks         2020 non-null   int64  
 7   Spend          2020 non-null   str    
 8   Conversions    1820 non-null   float64
 9   Active         2020 non-null   str    
 10  Clicks         40 non-null     float64
 11  Campaign_Tag   2020 non-null   str    
dtypes: float64(2), int64(2), str(8)
memory usage: 189.5 KB


In [179]:
data.head(10)

,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA
5,CMP-00006,Q4_BlackFriday_CMP-00006,2023-10-15 00:00:00,2023-10-28,Instagram,83386,2643,2697.03,NaN,1,NaN,IN
6,CMP-00007,Q3_Launch_CMP-00007,2023-10-07 00:00:00,2023-10-23,Facebook,38194,1135,1232.76,178.0,Yes,NaN,FA
7,CMP-00008,Q4_Launch_CMP-00008,2023-05-23,2023-05-28,Instagram,88498,1173,865.7,127.0,1,NaN,IN
8,CMP-00009,Q4_BlackFriday_CMP-00009,2023-03-23 00:00:00,2023-04-01,Google Ads,45131,1179,1046.18,104.0,1,NaN,GO
9,CMP-00010,Q2_Winter_CMP-00010,2023-03-21 00:00:00,2023-04-01,Email,61263,1153,1623.56,NaN,0,NaN,EM


# 1. Clean column names

In [180]:
data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_')

# # using dict and zip
# new_data_columns = data.columns.str.strip().str.lower().str.replace(' ', '_')
# data.rename(columns=dict(zip(data.columns, new_data_columns)), inplace=True)

# # using lambda function
# data.rename(columns=lambda x: x.strip().lower().replace(' ', '_'), inplace=True)

# 2. Type conversion

In [181]:
data_spend_dirty = data['spend'].str.contains(r'\$')
print(data.loc[data_spend_dirty, ['campaign_id', 'spend']].head(5))

data['spend'] = data['spend'].str.replace(r'[^\d.]', '', regex=True)
data['spend'] = pd.to_numeric(data['spend'], errors='coerce')

print('FIXED!!!')
print(data.loc[data_spend_dirty, ['campaign_id', 'spend']].head(5))

   campaign_id     spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22
31   CMP-00032  $2759.35
32   CMP-00033  $2393.02
FIXED!!!
   campaign_id    spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22
31   CMP-00032  2759.35
32   CMP-00033  2393.02


# 3. Categorical columns

In [182]:
print(data['channel'].unique())

cleanup_map = {
    'TikTok': 'TikTok',
    'Facebook': 'Facebook',
    'Email': 'Email',
    'Instagram': 'Instagram',
    'Google Ads': 'Google Ads',
    'E-mail': 'Email',
    'nan': np.nan,
    'Gogle': 'Google Ads',
    'Tik_Tok': 'TikTok',
    'Facebok': 'Facebook',
    'Insta_gram': 'Instagram'
}

data['channel'] = data['channel'].replace(cleanup_map)

print('FIXED!!!')
print(data['channel'].unique())

<StringArray>
[    'TikTok',   'Facebook',      'Email',  'Instagram', 'Google Ads',
     'E-mail',          nan,      'Gogle',    'Tik_Tok',    'Facebok',
 'Insta_gram']
Length: 11, dtype: str
FIXED!!!
<StringArray>
['TikTok', 'Facebook', 'Email', 'Instagram', 'Google Ads', nan]
Length: 6, dtype: str


# 4. BOOL columns

In [183]:
print(data['active'].unique())

bool_cleanup = {
    'Y': True,
    '0': False,
    'No': False,
    'True': True,
    'Yes': True,
    '1': True,
    'False': False
}

data['active'] = data['active'].replace(bool_cleanup)

print('FIXED!!!')
print(data['active'].unique())

<StringArray>
['Y', '0', 'No', 'True', 'Yes', '1', 'False']
Length: 7, dtype: str
FIXED!!!
[True False]


# 5. Datetime columns

In [187]:
data[['start_date', 'end_date']].value_counts()

start_date           end_date  
2023-08-03 00:00:00  2023-08-27    4
2023-08-31 00:00:00  2023-09-12    3
2023-05-27 00:00:00  2023-06-07    3
2023-06-02 00:00:00  2023-06-20    3
2023-06-19 00:00:00  2023-07-14    3
                                  ..
2023-06-25 00:00:00  2023-07-12    1
2023-12-06 00:00:00  2023-12-18    1
2023-12-31 00:00:00  2024-01-14    1
2023-10-31 00:00:00  2023-11-24    1
2023-02-19 00:00:00  2023-03-04    1
Name: count, Length: 1874, dtype: int64